# Large Prompt PDF RAG with Groq

This notebook matches `rag1_simple_pdf_large_prompt.py`.
It teaches how bigger instructions change the style and quality of the answer.

In [ ]:
!pip -q install openai python-dotenv numpy pypdf tiktoken fastembed

In [ ]:
import os
from getpass import getpass

import numpy as np
import tiktoken
from fastembed import TextEmbedding
from openai import OpenAI
from pypdf import PdfReader

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_file_path = next(iter(uploaded.keys())) if uploaded else "sample_ocr.pdf"
except ImportError:
    pdf_file_path = input("PDF path (default sample_ocr.pdf): ").strip() or "sample_ocr.pdf"

groq_api_key = os.getenv("GROQ_API_KEY") or getpass("Enter GROQ_API_KEY: ").strip()
chat_model = os.getenv("GROQ_CHAT_MODEL") or "qwen/qwen3.8-27b"
user_question = input("Question: ").strip() or "Review this invoice like an operations and finance analyst."

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)
embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text.strip())
    return "\n".join(pages)

def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> list[str]:
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    step = chunk_size - chunk_overlap
    return [tokenizer.decode(tokens[start:start + chunk_size]) for start in range(0, len(tokens), step)]

def get_embedding(text: str) -> list[float]:
    return list(next(embedder.embed([text])))

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def retrieve_best_chunk(chunks: list[str], question: str) -> str:
    chunk_embeddings = [get_embedding(chunk) for chunk in chunks]
    question_embedding = get_embedding(question)
    similarities = [cosine_similarity(question_embedding, chunk_embedding) for chunk_embedding in chunk_embeddings]
    return chunks[int(np.argmax(similarities))]

def ask_groq(retrieved_context: str, question: str) -> str:
    completion = client.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": "You are a senior operations, procurement, and finance analyst. Use only the retrieved context below. Start with a short plain-English summary. Then provide a bullet list of key commercial facts. Then provide a section called Checks and Concerns. In that section, point out OCR issues, unusual numbers, ambiguous wording, or anything that should be manually verified. If a fact is uncertain, say uncertain from context.\n\nRetrieved context:\n" + retrieved_context,
            },
            {"role": "user", "content": question},
        ],
    )
    return completion.choices[0].message.content or ""

In [ ]:
raw_pdf_text = extract_text_from_pdf(pdf_file_path)
chunks = chunk_text(raw_pdf_text)
retrieved_context = retrieve_best_chunk(chunks, user_question)
answer = ask_groq(retrieved_context, user_question)
print(answer)